In [1]:
pip install yfinance

In [4]:
import os
import time
import random
import math
from pathlib import Path

import numpy as np
import pandas as pd
import yfinance as yf

In [5]:
# =========================
# 0. 路径与参数设置
# =========================
from pathlib import Path

# 修正路径转义
INPUT_FILE = r"C:\Users\31314\Desktop\手动匹配错误公司并删除后.csv"

# 原始价格缓存目录：每个ticker一个csv
RAW_PRICE_DIR = Path("C:/Users/31314/Desktop/data/raw/prices")
RAW_PRICE_DIR.mkdir(parents=True, exist_ok=True)

# 处理后输出
PROCESSED_DIR = Path("C:/Users/31314/Desktop/data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# 日志文件
DOWNLOAD_LOG_FILE = PROCESSED_DIR / "C:/Users/31314/Desktop/price_download_log.csv"

# 下载参数
LOOKBACK_DAYS = 300
LOOKFORWARD_DAYS = 300

# 限流控制
BATCH_SIZE = 20              # 每批处理多少个ticker
SLEEP_BETWEEN_TICKERS = (1.5, 3.5)
SLEEP_BETWEEN_BATCHES = (5, 10)
MAX_RETRIES = 3

In [6]:
# =========================
# 1. 读取事件数据
# =========================
def read_input_csv(file_path: str) -> pd.DataFrame:
    """
    尝试多种常见中文编码读取CSV。
    """
    encodings = ["utf-8", "utf-8-sig", "gb18030", "gbk", "latin1"]
    last_error = None

    for enc in encodings:
        try:
            df = pd.read_csv(file_path, encoding=enc)
            print(f"[INFO] Successfully read file with encoding: {enc}")
            return df
        except Exception as e:
            last_error = e

    raise ValueError(f"Failed to read input CSV: {last_error}")


def prepare_event_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    清洗输入数据，保留执行第4步第1小点所需字段。
    """
    required_cols = ["company", "ticker", "announcement_date_verified"]
    missing_cols = [c for c in required_cols if c not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing required columns: {missing_cols}")

    work = df.copy()

    # 清理ticker
    work["ticker"] = work["ticker"].astype(str).str.strip()
    work = work[work["ticker"].notna()]
    work = work[work["ticker"] != ""]
    work = work[~work["ticker"].str.upper().eq("UNMATCHED")]

    # 日期转datetime
    work["announcement_date_verified"] = pd.to_datetime(
        work["announcement_date_verified"], errors="coerce"
    )

    # 删除无日期记录
    work = work.dropna(subset=["announcement_date_verified"])

    # 删除ticker缺失记录
    work = work.dropna(subset=["ticker"])

    # 去掉明显无效ticker
    work = work[~work["ticker"].isin(["nan", "None", "NONE"])]

    print(f"[INFO] Valid event rows after cleaning: {len(work)}")
    print(f"[INFO] Unique tickers to process: {work['ticker'].nunique()}")

    return work


# =========================
# 2. 按ticker聚合下载窗口
# =========================
def build_ticker_windows(events_df: pd.DataFrame) -> pd.DataFrame:
    """
    同一个ticker可能对应多个裁员事件。
    为减少Yahoo请求次数，按ticker聚合：
    start_date = 最早公告日 - 300天
    end_date   = 最晚公告日 + 300天
    """
    ticker_windows = (
        events_df.groupby("ticker")["announcement_date_verified"]
        .agg(min_event_date="min", max_event_date="max", event_count="count")
        .reset_index()
    )

    ticker_windows["start_date"] = ticker_windows["min_event_date"] - pd.Timedelta(days=LOOKBACK_DAYS)
    ticker_windows["end_date"] = ticker_windows["max_event_date"] + pd.Timedelta(days=LOOKFORWARD_DAYS)

    return ticker_windows


# =========================
# 3. 下载单个ticker价格数据
# =========================
def safe_filename(ticker: str) -> str:
    """
    处理少数ticker里可能存在的不适合作为文件名的字符。
    """
    return (
        ticker.replace("/", "_")
              .replace("\\", "_")
              .replace(":", "_")
              .replace("*", "_")
              .replace("?", "_")
              .replace('"', "_")
              .replace("<", "_")
              .replace(">", "_")
              .replace("|", "_")
    )


def download_one_ticker(ticker: str, start_date: pd.Timestamp, end_date: pd.Timestamp) -> dict:
    """
    下载单个ticker的日度价格数据，保存到本地csv。
    返回状态字典用于日志记录。
    """
    out_file = RAW_PRICE_DIR / f"{safe_filename(ticker)}.csv"

    # 已存在则直接跳过，避免重复下载
    if out_file.exists():
        try:
            cached_df = pd.read_csv(out_file)
            return {
                "ticker": ticker,
                "status": "cached",
                "rows": len(cached_df),
                "start_date": start_date.strftime("%Y-%m-%d"),
                "end_date": end_date.strftime("%Y-%m-%d"),
                "message": "File already exists"
            }
        except Exception:
            # 文件存在但坏了，继续重新下载
            pass

    last_error = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            # yfinance 的 end 是开区间，所以 +1 天
            yf_end = end_date + pd.Timedelta(days=1)

            data = yf.download(
                tickers=ticker,
                start=start_date.strftime("%Y-%m-%d"),
                end=yf_end.strftime("%Y-%m-%d"),
                progress=False,
                auto_adjust=False,   # 保留Adj Close，便于明确使用复权价
                threads=False
            )

            if data is None or data.empty:
                return {
                    "ticker": ticker,
                    "status": "empty",
                    "rows": 0,
                    "start_date": start_date.strftime("%Y-%m-%d"),
                    "end_date": end_date.strftime("%Y-%m-%d"),
                    "message": "No data returned from Yahoo Finance"
                }

            # 有时候单ticker也可能返回MultiIndex，统一处理
            if isinstance(data.columns, pd.MultiIndex):
                data.columns = data.columns.get_level_values(0)

            data = data.reset_index()

            # 统一列名检查
            if "Date" not in data.columns:
                return {
                    "ticker": ticker,
                    "status": "bad_format",
                    "rows": len(data),
                    "start_date": start_date.strftime("%Y-%m-%d"),
                    "end_date": end_date.strftime("%Y-%m-%d"),
                    "message": "Downloaded data missing Date column"
                }

            # 优先使用 Adj Close 作为复权收盘价
            if "Adj Close" in data.columns:
                data["adj_close"] = data["Adj Close"]
            elif "Close" in data.columns:
                # 极少数情况下 Adj Close 缺失，退而求其次
                data["adj_close"] = data["Close"]
            else:
                return {
                    "ticker": ticker,
                    "status": "no_price_column",
                    "rows": len(data),
                    "start_date": start_date.strftime("%Y-%m-%d"),
                    "end_date": end_date.strftime("%Y-%m-%d"),
                    "message": "Neither Adj Close nor Close found"
                }

            data = data.rename(columns={"Date": "date"})

            # 保留常用列；不存在的列补空
            keep_cols = ["date", "Open", "High", "Low", "Close", "Adj Close", "Volume", "adj_close"]
            for col in keep_cols:
                if col not in data.columns:
                    data[col] = np.nan

            data = data[keep_cols].copy()
            data["ticker"] = ticker

            # 日期排序
            data["date"] = pd.to_datetime(data["date"], errors="coerce")
            data = data.dropna(subset=["date"]).sort_values("date").reset_index(drop=True)

            # 计算日对数收益率 ret = log(P_t / P_{t-1})
            data["ret"] = np.log(data["adj_close"] / data["adj_close"].shift(1))

            # 输出列顺序
            final_cols = [
                "date", "ticker", "Open", "High", "Low", "Close", "Adj Close",
                "adj_close", "Volume", "ret"
            ]
            data = data[final_cols]

            # 保存
            data.to_csv(out_file, index=False)

            return {
                "ticker": ticker,
                "status": "ok",
                "rows": len(data),
                "start_date": start_date.strftime("%Y-%m-%d"),
                "end_date": end_date.strftime("%Y-%m-%d"),
                "message": ""
            }

        except Exception as e:
            last_error = str(e)
            wait_seconds = 5 * attempt + random.uniform(1, 3)
            print(f"[WARN] {ticker} attempt {attempt} failed: {last_error}")
            print(f"[INFO] Sleeping {wait_seconds:.1f}s before retry...")
            time.sleep(wait_seconds)

    return {
        "ticker": ticker,
        "status": "failed",
        "rows": np.nan,
        "start_date": start_date.strftime("%Y-%m-%d"),
        "end_date": end_date.strftime("%Y-%m-%d"),
        "message": last_error if last_error else "Unknown error"
    }


In [ ]:
# =========================
# 5. 主程序
# =========================
def main():
    print("=" * 60)
    print("Step 15: Download adjusted daily close prices and compute log returns")
    print("=" * 60)

    # 读取原始事件数据
    df = read_input_csv(INPUT_FILE)
    events_df = prepare_event_data(df)

    # 构建每个ticker的下载区间
    ticker_windows = build_ticker_windows(events_df)
    print(f"[INFO] Tickers to download: {len(ticker_windows)}")

    results = []

    # 分批处理，减轻Yahoo限流
    ticker_windows = ticker_windows.sort_values("ticker").reset_index(drop=True)

    for batch_start in range(0, len(ticker_windows), BATCH_SIZE):
        batch_end = min(batch_start + BATCH_SIZE, len(ticker_windows))
        batch_df = ticker_windows.iloc[batch_start:batch_end]

        print(f"\n[INFO] Processing batch {batch_start + 1} to {batch_end} ...")

        for _, row in batch_df.iterrows():
            ticker = row["ticker"]
            start_date = row["start_date"]
            end_date = row["end_date"]

            print(f"[INFO] Downloading {ticker} | {start_date.date()} -> {end_date.date()}")
            result = download_one_ticker(ticker, start_date, end_date)
            results.append(result)

            # ticker之间随机暂停
            sleep_time = random.uniform(*SLEEP_BETWEEN_TICKERS)
            time.sleep(sleep_time)

        # batch之间再暂停久一点
        if batch_end < len(ticker_windows):
            batch_sleep = random.uniform(*SLEEP_BETWEEN_BATCHES)
            print(f"[INFO] Batch finished. Sleeping {batch_sleep:.1f}s ...")
            time.sleep(batch_sleep)

    # 保存下载日志
    log_df = pd.DataFrame(results)
    log_df.to_csv(DOWNLOAD_LOG_FILE, index=False)
    print(f"\n[INFO] Download log saved to: {DOWNLOAD_LOG_FILE}")



if __name__ == "__main__":
    main()

Step 15: Download adjusted daily close prices and compute log returns
[INFO] Successfully read file with encoding: gb18030
[INFO] Valid event rows after cleaning: 779
[INFO] Unique tickers to process: 491
[INFO] Tickers to download: 491

[INFO] Processing batch 1 to 20 ...
[INFO] Downloading 0020.HK | 2022-11-01 -> 2024-06-23



1 Failed download:
['0020.HK']: YFRateLimitError('Too Many Requests. Rate limited. Try after a while.')


[INFO] Downloading 005930.KS | 2022-05-18 -> 2024-01-08



1 Failed download:
['005930.KS']: YFRateLimitError('Too Many Requests. Rate limited. Try after a while.')


[INFO] Downloading 036570.KS | 2022-04-08 -> 2023-11-29


...

[INFO] Downloading LCID | 2022-06-01 -> 2026-12-17



1 Failed download:
['LCID']: YFRateLimitError('Too Many Requests. Rate limited. Try after a while.')


[INFO] Downloading LDTCW | 2024-07-26 -> 2026-03-18



1 Failed download:
['LDTCW']: YFRateLimitError('Too Many Requests. Rate limited. Try after a while.')


[INFO] Batch finished. Sleeping 5.3s ...


In [6]:
#把所有单独csv文件合并成起来

from pathlib import Path
import pandas as pd

RAW_PRICE_DIR = Path("C:/Users/31314/Desktop/data/raw/prices")
PROCESSED_DIR = Path("C:/Users/31314/Desktop/data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

def combine_all_price_files(output_file: Path):
    files = list(RAW_PRICE_DIR.glob("*.csv"))
    if not files:
        print("[WARN] No raw price files found in C:/Users/31314/Desktop/data/raw/prices/")
        return pd.DataFrame()

    all_dfs = []
    for f in files:
        try:
            tmp = pd.read_csv(f)
            all_dfs.append(tmp)
        except Exception as e:
            print(f"[WARN] Failed to read {f.name}: {e}")

    if not all_dfs:
        print("[WARN] No valid raw price files to combine.")
        return pd.DataFrame()

    combined = pd.concat(all_dfs, ignore_index=True)
    combined["date"] = pd.to_datetime(combined["date"], errors="coerce")
    combined = combined.dropna(subset=["date"])
    combined = combined.sort_values(["ticker", "date"]).reset_index(drop=True)

    output_file.parent.mkdir(parents=True, exist_ok=True)
    combined.to_csv(output_file, index=False)

    print(f"[INFO] Combined price panel saved to: {output_file}")
    print(f"[INFO] Total rows: {len(combined)}")
    print(f"[INFO] Unique tickers: {combined['ticker'].nunique()}")

    return combined

if __name__ == "__main__":
    combine_all_price_files(Path("C:/Users/31314/Desktop/data/processed/已下载股票价格数据.csv"))

[INFO] Combined price panel saved to: data/processed/price_panel_raw.csv
[INFO] Total rows: 224974
[INFO] Unique tickers: 390
